# 🏥 MedAssist AI — Multimodal Medical RAG Assistant (v2)
**Using MedGemma 4B-IT | LangChain + ChromaDB | FastAPI + ngrok**

---

⚠️ **Disclaimer:** MedGemma is used as a research and developer model for educational decision-support prototyping. This is NOT a clinical diagnostic tool.

---

### 📋 Notebook Sections
1. Mount Google Drive & Install Dependencies
2. HuggingFace Login & MedGemma Access Check
3. Load MedGemma 4B-IT (from Drive cache or HuggingFace)
4. Setup Embedding Model (MiniLM)
5. Setup ChromaDB Vector Store
6. Build RAG Pipeline
7. Core Chat Function
8. FastAPI Backend (with enhanced endpoints)
9. Launch with ngrok

**Runtime Required:** T4 GPU (Go to Runtime → Change runtime type → T4 GPU)

**First run:** Downloads model from HuggingFace and saves to Google Drive (~8.6 GB, takes 3-5 min)

**Subsequent runs:** Loads directly from Google Drive (~1-2 min, no download needed)

### API Endpoints
| Method | Endpoint | Purpose |
|--------|----------|---------|
| GET | /health | Connection status |
| GET | /model-info | Model stats, VRAM, quantization |
| GET | /suggestions | Sample questions for quick start |
| POST | /chat | Send message + optional image |
| POST | /upload-pdf | Upload and index a PDF |
| DELETE | /clear-pdf | Remove document and clear index |

## 1. Mount Google Drive & Install Dependencies

In [ ]:
# Mount Google Drive for model caching
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted!")

Mounted at /content/drive
✅ Google Drive mounted!


In [ ]:
!pip install -q \
    transformers>=4.45.0 \
    accelerate \
    bitsandbytes \
    torch \
    Pillow \
    langchain \
    langchain-community \
    langchain-text-splitters \
    pypdf \
    sentence-transformers \
    chromadb \
    fastapi \
    uvicorn \
    python-multipart \
    pyngrok \
    nest-asyncio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
google-adk 1.27.1 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.40.0 which is incompatible.
google-adk 1.27.1 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.40.0 which is incompatib

## 2. HuggingFace Login & MedGemma Access Check

**First time only** — needed to download the model. After the model is saved to Google Drive, you can skip this cell on future runs.

MedGemma is a **gated model**. You need to:
1. Go to [google/medgemma-4b-it](https://huggingface.co/google/medgemma-4b-it)
2. Click **"Agree and access repository"** (if you haven't already)
3. Create a HuggingFace token at [Settings → Tokens](https://huggingface.co/settings/tokens) with **Read** access
4. Paste it below when prompted

In [ ]:
import os

# Updated to point to the CongiMed folder on your Drive
MODEL_DRIVE_DIR = "/content/drive/MyDrive/CongiMed/medgemma-4b-it-4bit"
MODEL_CACHED = os.path.exists(os.path.join(MODEL_DRIVE_DIR, "config.json"))

if MODEL_CACHED:
    print("✅ Model found on Google Drive! No HuggingFace login needed.")
    print(f"   Cache location: {MODEL_DRIVE_DIR}")
    print("   Skipping login — proceed to the next cell.")
else:
    print("📥 Model NOT found on Google Drive. First-time setup needed.")
    print("   Logging in to HuggingFace...")
    from huggingface_hub import login, model_info
    login()

    # Check access
    try:
        info = model_info("google/medgemma-4b-it")
        print(f"✅ Access confirmed! Model: {info.modelId}")
    except Exception as e:
        if "401" in str(e) or "403" in str(e):
            print("❌ Access DENIED. Please go to https://huggingface.co/google/medgemma-4b-it")
            print("   and click 'Agree and access repository' first.")
        else:
            print(f"❌ Error: {e}")

📥 Model NOT found on Google Drive. First-time setup needed.
   Logging in to HuggingFace...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Access confirmed! Model: google/medgemma-4b-it


## 3. Load MedGemma 4B-IT (4-bit Quantized)

This cell automatically:
- **If cached on Drive:** Loads from Google Drive (~1-2 min, no internet needed)
- **If not cached:** Downloads from HuggingFace, loads, then saves to Drive for next time

In [ ]:
import torch
import time
import os
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = "google/medgemma-4b-it"
MODEL_DRIVE_DIR = "/content/drive/MyDrive/CongiMed/medgemma-4b-it-4bit"
CONFIG_PATH = os.path.join(MODEL_DRIVE_DIR, "config.json")

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

load_start = time.time()

# Check if model is already saved to Drive
if os.path.exists(CONFIG_PATH):
    print("📂 Loading MedGemma from Google Drive cache...")
    processor = AutoProcessor.from_pretrained(MODEL_DRIVE_DIR)
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_DRIVE_DIR,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )
    loaded_from = "google_drive"
else:
    print("📥 Model not found on Drive. Downloading from HuggingFace...")
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )

    # Save to Drive for future use
    print(f"💾 Saving model to {MODEL_DRIVE_DIR} (this may take a few minutes)... ")
    os.makedirs(MODEL_DRIVE_DIR, exist_ok=True)
    processor.save_pretrained(MODEL_DRIVE_DIR)
    model.save_pretrained(MODEL_DRIVE_DIR)
    loaded_from = "huggingface"

load_time = time.time() - load_start

# Store model metadata
model_metadata = {
    "model_id": MODEL_ID,
    "gpu_name": torch.cuda.get_device_name(0),
    "vram_gb": round(torch.cuda.memory_allocated() / 1024**3, 2),
    "load_time_seconds": round(load_time, 1),
    "loaded_from": loaded_from,
    "quantization": "4-bit",
    "compute_dtype": "bfloat16"
}

print(f"\n{'='*50}")
print(f"✅ MedGemma ready!")
print(f"   Source: {loaded_from.upper()}")
print(f"   Load time: {model_metadata['load_time_seconds']}s")
print(f"{'='*50}")

📥 First time setup — downloading from HuggingFace...
   This takes ~3-5 minutes. Model will be saved to Drive after.

   ⏳ Loading processor...


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

   ✅ Processor loaded
   ⏳ Loading model (4-bit quantized)...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

### Quick Test — Verify Model Works

In [ ]:
# Quick sanity check with a simple medical question
test_messages = [
    {"role": "system", "content": [{"type": "text", "text": "You are a helpful medical assistant."}]},
    {"role": "user", "content": [{"type": "text", "text": "What are the common symptoms of pneumonia?"}]}
]

inputs = processor.apply_chat_template(
    test_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.7)

response = processor.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("🧪 Test response:")
print(response)

## 4. Setup Embedding Model (MiniLM)

This runs on CPU — no GPU needed. Used for encoding document chunks and user queries for RAG retrieval.

In [ ]:
from sentence_transformers import SentenceTransformer

# Updated to point to the CongiMed folder on your Drive
EMBEDDING_DRIVE_DIR = "/content/drive/MyDrive/CongiMed/all-MiniLM-L6-v2"

if os.path.exists(os.path.join(EMBEDDING_DRIVE_DIR, "config.json")):
    print("📂 Loading MiniLM from Google Drive cache...")
    embedding_model = SentenceTransformer(EMBEDDING_DRIVE_DIR)
else:
    print("📥 Downloading MiniLM embedding model...")
    embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    print("💾 Saving to Google Drive...")
    os.makedirs(EMBEDDING_DRIVE_DIR, exist_ok=True)
    embedding_model.save(EMBEDDING_DRIVE_DIR)

print(f"✅ MiniLM loaded! Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

# Quick test
test_embedding = embedding_model.encode("chest x-ray shows bilateral opacity")
print(f"   Test embedding shape: {test_embedding.shape}")

## 5. Setup ChromaDB Vector Store

In [ ]:
import chromadb
import shutil

# Keep it temporary! This wipes cleanly when Colab restarts.
CHROMA_DIR = "/content/chroma_db"

# Initialize ChromaDB
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

COLLECTION_NAME = "medical_docs"

def get_or_create_collection():
    """Get existing collection or create a new one."""
    return chroma_client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"}
    )

def clear_collection():
    """Delete and recreate the collection."""
    try:
        chroma_client.delete_collection(COLLECTION_NAME)
    except:
        pass
    return get_or_create_collection()

collection = get_or_create_collection()
print(f"✅ Temporary ChromaDB ready! Collection: {COLLECTION_NAME}")
print(f"   Current documents: {collection.count()}")

## 6. Build RAG Pipeline

PDF ingestion → chunking → embedding → storage → retrieval

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid
import tempfile

# Text splitter config from design docs
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Track if a PDF is currently loaded
pdf_loaded = {"status": False, "filename": None, "pages": 0, "chunks": 0}


def ingest_pdf(file_bytes: bytes, filename: str) -> dict:
    """
    Process a PDF: extract text, chunk it, embed it, store in ChromaDB.
    Returns status dict with pages_indexed and chunks_created.
    """
    global collection, pdf_loaded

    # Clear any existing documents first
    collection = clear_collection()

    # Save PDF to temp file for PyPDFLoader
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
        tmp.write(file_bytes)
        tmp_path = tmp.name

    try:
        ingest_start = time.time()

        # Extract text page by page
        loader = PyPDFLoader(tmp_path)
        pages = loader.load()
        print(f"   📄 Extracted {len(pages)} pages from {filename}")

        # Chunk the documents
        chunks = text_splitter.split_documents(pages)
        print(f"   🔪 Split into {len(chunks)} chunks")

        if len(chunks) == 0:
            return {"status": "error", "message": "No text could be extracted from this PDF."}

        # Embed and store each chunk
        texts = [chunk.page_content for chunk in chunks]
        metadatas = [
            {
                "page": chunk.metadata.get("page", 0),
                "source": filename
            }
            for chunk in chunks
        ]
        ids = [str(uuid.uuid4()) for _ in chunks]

        # Encode all chunks with MiniLM
        embeddings = embedding_model.encode(texts).tolist()

        # Add to ChromaDB
        collection.add(
            documents=texts,
            embeddings=embeddings,
            metadatas=metadatas,
            ids=ids
        )

        ingest_time = round(time.time() - ingest_start, 1)

        pdf_loaded["status"] = True
        pdf_loaded["filename"] = filename
        pdf_loaded["pages"] = len(pages)
        pdf_loaded["chunks"] = len(chunks)

        print(f"   ✅ Indexed {len(chunks)} chunks into ChromaDB in {ingest_time}s")
        return {
            "status": "success",
            "filename": filename,
            "pages_indexed": len(pages),
            "chunks_created": len(chunks),
            "indexing_time_seconds": ingest_time
        }

    finally:
        os.unlink(tmp_path)


def retrieve_context(query: str, n_results: int = 3) -> list:
    """
    Retrieve top-N relevant chunks from ChromaDB for the given query.
    Returns list of dicts with 'text', 'page', and 'relevance_score' keys.
    """
    if not pdf_loaded["status"] or collection.count() == 0:
        return []

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(n_results, collection.count()),
        include=["documents", "metadatas", "distances"]
    )

    citations = []
    for i in range(len(results["documents"][0])):
        citations.append({
            "text": results["documents"][0][i],
            "page": results["metadatas"][0][i].get("page", "unknown"),
            "relevance_score": round(1 - results["distances"][0][i], 3)  # cosine similarity
        })

    return citations


print("✅ RAG pipeline ready!")

## 7. Core Chat Function

Handles text-only and multimodal (image + text) queries, with optional RAG context.
Returns response text, citations, inference time, and token count.

In [ ]:
from PIL import Image
import io
import traceback
import time
import torch

def build_prompt(user_message: str, conversation_history: list = None,
                 image: Image.Image = None, rag_citations: list = None) -> list:
    """
    Build the message list for MedGemma.
    All content fields use the list-of-dicts format required by the processor.
    """
    messages = []

    # System prompt
    if rag_citations:
        context_str = "\n\n".join(
            f"[Source: Page {c['page']}] {c['text']}" for c in rag_citations
        )
        system_text = (
            "You are a medical assistant. Answer the user's question based on the "
            "provided document context below. If the context doesn't contain relevant "
            "information, say so clearly.\n\n"
            f"Document Context:\n{context_str}"
        )
    else:
        system_text = (
            "You are a helpful medical assistant. Provide informative responses "
            "to medical questions. Always remind the user that this is for "
            "educational purposes only and not a substitute for professional medical advice."
        )

    messages.append({"role": "system", "content": [{"type": "text", "text": system_text}]})

    # Add conversation history (last 6 turns to stay within context limits)
    if conversation_history:
        for msg in conversation_history[-6:]:
            messages.append({
                "role": msg["role"],
                "content": [{"type": "text", "text": msg["content"]}]
            })

    # Build user message content
    if image:
        user_content = [
            {"type": "image", "image": image},
            {"type": "text", "text": user_message}
        ]
    else:
        user_content = [{"type": "text", "text": user_message}]

    messages.append({"role": "user", "content": user_content})

    return messages


def generate_response(user_message: str, conversation_history: list = None,
                      image: Image.Image = None, top_k: int = 3) -> dict:
    """
    Main chat function. Handles RAG retrieval, prompt building, and generation.
    Returns dict with 'response', 'citations', 'inference_time_ms', and 'tokens_generated'.
    """
    try:
        # Step 1: RAG retrieval with threaded top_k
        citations = retrieve_context(user_message, n_results=top_k) if pdf_loaded["status"] else []

        # Step 2: Build prompt
        messages = build_prompt(
            user_message=user_message,
            conversation_history=conversation_history,
            image=image,
            rag_citations=citations if citations else None
        )

        # Step 3: Tokenize
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        input_token_count = inputs["input_ids"].shape[1]

        # Step 4: Generate with timing
        gen_start = time.time()

        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
            )

        gen_time = time.time() - gen_start

        # Step 5: Decode (strip the prompt tokens)
        output_tokens = output[0][input_token_count:]
        tokens_generated = len(output_tokens)

        response_text = processor.decode(
            output_tokens,
            skip_special_tokens=True
        )

        return {
            "response": response_text.strip(),
            "citations": citations,
            "inference_time_ms": round(gen_time * 1000),
            "tokens_generated": tokens_generated,
            "input_tokens": input_token_count,
            "tokens_per_second": round(tokens_generated / gen_time, 1) if gen_time > 0 else 0
        }

    except Exception as e:
        traceback.print_exc()
        return {
            "response": f"Error generating response: {str(e)}",
            "citations": [],
            "inference_time_ms": 0,
            "tokens_generated": 0,
            "input_tokens": 0,
            "tokens_per_second": 0
        }

print("✅ build_prompt and generate_response defined!")

### Quick Test — Chat without PDF

In [ ]:
result = generate_response("What is a normal resting heart rate for an adult?")
print("Response:", result["response"])
print(f"\n⏱️  Inference: {result['inference_time_ms']}ms")
print(f"📊 Tokens: {result['tokens_generated']} generated ({result['tokens_per_second']} tok/s)")
print(f"📄 Citations: {result['citations']}")

## 8. FastAPI Backend

### Endpoints:
| Method | Endpoint | Purpose |
|--------|----------|---------|
| GET | `/health` | Connection check + PDF status |
| GET | `/model-info` | GPU, VRAM, quantization, model details |
| GET | `/suggestions` | Sample questions for quick start |
| POST | `/chat` | Message + optional image + conversation history |
| POST | `/upload-pdf` | Upload and index a PDF |
| DELETE | `/clear-pdf` | Remove document and clear index |

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from typing import Optional
import json
import base64
import time
import io
from PIL import Image
from datetime import datetime

app = FastAPI(title="MedAssist AI", version="2.0")

# CORS — allow React frontend on Vercel
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # In production, restrict to your Vercel domain
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Track server uptime
server_start_time = time.time()


# ==================== GET ENDPOINTS ====================

@app.get("/health")
async def health_check():
    """Check if backend is online and get PDF status."""
    return {
        "status": "ok",
        "model": MODEL_ID,
        "pdf_loaded": pdf_loaded["status"],
        "pdf_filename": pdf_loaded["filename"],
        "pdf_chunks": pdf_loaded["chunks"],
        "uptime_seconds": round(time.time() - server_start_time)
    }


@app.get("/model-info")
async def get_model_info():
    """Return model details, VRAM usage, and GPU info for frontend display."""
    import torch
    return {
        "model_id": model_metadata["model_id"],
        "quantization": model_metadata["quantization"],
        "compute_dtype": model_metadata["compute_dtype"],
        "gpu_name": model_metadata["gpu_name"],
        "vram_used_gb": round(torch.cuda.memory_allocated() / 1024**3, 2),
        "vram_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if torch.cuda.is_available() else 0,
        "vram_utilization_percent": round((torch.cuda.memory_allocated() / torch.cuda.get_device_properties(0).total_memory) * 100, 1) if torch.cuda.is_available() else 0,
        "model_load_time_seconds": model_metadata["load_time_seconds"],
        "loaded_from": model_metadata["loaded_from"],
        "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
        "embedding_dimension": 384,
        "vector_store": "ChromaDB",
        "max_new_tokens": 512
    }


@app.get("/suggestions")
async def get_suggestions():
    """Return sample questions for the frontend quick-start buttons."""
    general_questions = [
        {"text": "What are the common symptoms of pneumonia?", "category": "general", "icon": "🫁"},
        {"text": "Explain the difference between Type 1 and Type 2 diabetes.", "category": "general", "icon": "🩸"},
        {"text": "What does a complete blood count (CBC) test measure?", "category": "general", "icon": "🧪"},
        {"text": "What are the risk factors for cardiovascular disease?", "category": "general", "icon": "❤️"}
    ]

    pdf_questions = [
        {"text": "Summarize the key findings in this document.", "category": "pdf", "icon": "📄"},
        {"text": "What are the abnormal values in this report?", "category": "pdf", "icon": "⚠️"},
        {"text": "Explain the diagnosis mentioned in this document.", "category": "pdf", "icon": "🔍"}
    ]

    image_questions = [
        {"text": "What do you observe in this medical image?", "category": "image", "icon": "🖼️"},
        {"text": "Are there any abnormalities visible in this X-ray?", "category": "image", "icon": "🩻"}
    ]

    return {
        "general": general_questions,
        "pdf": pdf_questions,
        "image": image_questions,
        "pdf_active": pdf_loaded["status"]
    }


# ==================== POST/DELETE ENDPOINTS ====================

@app.post("/upload-pdf")
async def upload_pdf(file: UploadFile = File(...)):
    """Upload and index a PDF for RAG."""
    if not file.filename.lower().endswith(".pdf"):
        raise HTTPException(status_code=400, detail="Only PDF files are supported.")

    file_bytes = await file.read()

    if len(file_bytes) == 0:
        raise HTTPException(status_code=400, detail="Empty file uploaded.")

    if len(file_bytes) > 50 * 1024 * 1024:  # 50MB limit
        raise HTTPException(status_code=400, detail="File too large. Maximum size is 50MB.")

    print(f"📥 Received PDF: {file.filename} ({len(file_bytes) / 1024:.1f} KB)")
    result = ingest_pdf(file_bytes, file.filename)

    if result["status"] == "error":
        raise HTTPException(status_code=400, detail=result["message"])

    return result


@app.delete("/clear-pdf")
async def clear_pdf():
    """Remove uploaded document and safely clear vector index."""
    global collection, pdf_loaded

    # 1. Blind the system to prevent race condition
    pdf_loaded["status"] = False

    try:
        # 2. Destroy and recreate
        chroma_client.delete_collection("medical_docs")
        collection = chroma_client.create_collection(
            name="medical_docs",
            metadata={"hnsw:space": "cosine"}
        )
        pdf_loaded = {"status": False, "filename": None, "pages": 0, "chunks": 0}
        return {"status": "success", "message": "Document cache flushed."}
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Flush failed: {str(e)}")


@app.post("/reset-session")
async def reset_session():
    """Hard wipe of the entire clinical context to prevent patient data bleed."""
    global collection, pdf_loaded

    # 1. Blind the system to prevent race condition
    pdf_loaded["status"] = False

    try:
        # 2. Destroy and recreate
        chroma_client.delete_collection("medical_docs")
        collection = chroma_client.create_collection(
            name="medical_docs",
            metadata={"hnsw:space": "cosine"}
        )
        pdf_loaded = {"status": False, "filename": None, "pages": 0, "chunks": 0}
        return {"status": "success", "message": "Clinical context and vector index purged."}
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Reset failed: {str(e)}")


@app.post("/chat")
async def chat(
    message: str = Form(...),
    history: str = Form(default="[]"),
    image: Optional[UploadFile] = File(default=None),
    top_k: int = Form(default=3) # ADDED top_k
):
    """Send a message with optional image attachment. Returns AI response with metadata."""
    # Parse conversation history
    try:
        conversation_history = json.loads(history)
    except json.JSONDecodeError:
        conversation_history = []

    # Process image if provided
    pil_image = None
    if image and image.filename:
        if image.content_type not in ["image/jpeg", "image/png"]:
            raise HTTPException(status_code=400, detail="Only JPEG and PNG images are supported.")

        image_bytes = await image.read()

        if len(image_bytes) > 10 * 1024 * 1024:  # 10MB limit
            raise HTTPException(status_code=400, detail="Image too large. Maximum size is 10MB.")

        pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        print(f"🖼️  Image attached: {image.filename} ({pil_image.size})")

    print(f"💬 Message: {message[:100]}..." if len(message) > 100 else f"💬 Message: {message}")

    # Generate response with top_k threaded down
    result = generate_response(
        user_message=message,
        conversation_history=conversation_history,
        image=pil_image,
        top_k=top_k
    )

    return {
        "response": result["response"],
        "citations": result["citations"],
        "pdf_active": pdf_loaded["status"],
        "inference_time_ms": result["inference_time_ms"],
        "tokens_generated": result["tokens_generated"],
        "input_tokens": result["input_tokens"],
        "tokens_per_second": result["tokens_per_second"]
    }


@app.post("/export-report")
async def export_report(history: str = Form(...)):
    """Forces MedGemma to synthesize the chat history into a structured clinical document."""
    import torch
    try:
        conversation_history = json.loads(history)
    except Exception:
        raise HTTPException(status_code=400, detail="Invalid history format")

    if not conversation_history:
         return {"report": "No clinical data to export."}

    raw_transcript = "\n".join([f"{msg['role'].upper()}: {msg['content']}" for msg in conversation_history])

    # Hardened system prompt preventing preambles
    synthesis_prompt = (
        "You are an expert clinical summarizer. Output ONLY the structured report with no preamble or commentary. "
        "Review the following diagnostic transcript and generate a formal, structured medical report with exactly these sections:\n\n"
        "1. CHIEF COMPLAINT/QUERY\n"
        "2. KEY CLINICAL FINDINGS\n"
        "3. DIFFERENTIAL ANALYSIS\n"
        "4. CITED RAG SOURCES\n\n"
        f"TRANSCRIPT:\n{raw_transcript}"
    )

    inputs = processor.apply_chat_template(
        [{"role": "user", "content": [{"type": "text", "text": synthesis_prompt}]}],
        add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    # OOM-Hardened Inference Block
    try:
        with torch.inference_mode():
            output = model.generate(**inputs, max_new_tokens=1024, temperature=0.2)

        report_text = processor.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

        # Exact payload match for React
        return {
            "report": report_text.strip(),
            "case_id": "4882-QX",
            "timestamp": datetime.now().strftime("%Y%m%d_%H%M%S"),
            "filename": pdf_loaded.get("filename", "unknown")
        }
    except torch.cuda.OutOfMemoryError:
        raise HTTPException(status_code=503, detail="GPU memory exhausted. Try again after clearing session.")
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Report synthesis failed: {str(e)}")


print("✅ FastAPI app created with all endpoints!")
print("   Endpoints: /health, /model-info, /suggestions, /chat, /upload-pdf, /clear-pdf, /reset-session, /export-report")

## 9. Launch Server with ngrok

This exposes your Colab backend to the internet so the React frontend on Vercel can connect.

**You need a free ngrok account:**
1. Sign up at [ngrok.com](https://ngrok.com)
2. Go to **Your Authtoken** page
3. Copy your auth token and paste it below

In [ ]:
# Set your ngrok auth token
NGROK_AUTH_TOKEN = ""  # <-- Paste your ngrok token here

if not NGROK_AUTH_TOKEN:
    NGROK_AUTH_TOKEN = input("Enter your ngrok auth token: ").strip()

In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn

# Allow nested async (needed for Colab)
nest_asyncio.apply()

# Set ngrok auth
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Kill any existing tunnels
ngrok.kill()

# Start ngrok tunnel
PORT = 8000
public_url = ngrok.connect(PORT)

print("=" * 60)
print(f"🚀 MedAssist AI Backend v2 is LIVE!")
print(f"=" * 60)
print(f"")
print(f"   🌐 Public URL:   {public_url}")
print(f"")
print(f"   📋 Endpoints:")
print(f"      GET  {public_url}/health")
print(f"      GET  {public_url}/model-info")
print(f"      GET  {public_url}/suggestions")
print(f"      POST {public_url}/chat")
print(f"      POST {public_url}/upload-pdf")
print(f"      DEL  {public_url}/clear-pdf")
print(f"")
print(f"   Set this as VITE_API_URL in your React .env file.")
print(f"   ⚠️  This URL changes every time you restart ngrok!")
print(f"=" * 60)

# Run FastAPI server (this blocks — the server keeps running)
uvicorn.run(app, host="0.0.0.0", port=PORT)

---

## 🧪 Testing Commands

Use `curl` or Postman with your ngrok URL to test.

### Health Check
```bash
curl YOUR_NGROK_URL/health
```

### Model Info
```bash
curl YOUR_NGROK_URL/model-info
```

### Suggestions
```bash
curl YOUR_NGROK_URL/suggestions
```

### Chat (text only)
```bash
curl -X POST YOUR_NGROK_URL/chat \
  -F "message=What are the symptoms of diabetes?" \
  -F 'history=[]'
```

### Upload PDF
```bash
curl -X POST YOUR_NGROK_URL/upload-pdf \
  -F "file=@/path/to/your/medical_report.pdf"
```

### Chat with image
```bash
curl -X POST YOUR_NGROK_URL/chat \
  -F "message=What do you see in this chest X-ray?" \
  -F 'history=[]' \
  -F "image=@/path/to/xray.jpg"
```

### Clear PDF
```bash
curl -X DELETE YOUR_NGROK_URL/clear-pdf
```

### Example /chat Response
```json
{
  "response": "A normal resting heart rate for adults is 60-100 bpm...",
  "citations": [
    {"text": "chunk text...", "page": 3, "relevance_score": 0.847}
  ],
  "pdf_active": true,
  "inference_time_ms": 12340,
  "tokens_generated": 187,
  "input_tokens": 524,
  "tokens_per_second": 15.2
}
```